# ⚡ SQLCoder-Lite: Enterprise Text-to-SQL & Agentic BI Assistant
### Fine-Tuning Mistral-7B (4-Bit QLoRA) on `b-mc2/sql-create-context` with Live Database Execution & Executive AI Insights
**Author:** Sahil Goury | **Stack:** Unsloth, Hugging Face, TRL, SQLite3, Gradio | **Hardware:** Free Google Colab Tesla T4 GPU (16GB VRAM)

---

### 🌟 Project Overview & Architecture:
This project fine-tunes **Mistral-7B-v0.3** into an enterprise-grade Text-to-SQL & Business Intelligence (BI) Analyst agent using **4-bit NF4 Quantization** and **QLoRA via Unsloth**.
```
[User Natural Query] ──► [Mistral-7B (QLoRA)] ──► [Valid SQL] ──► [SQLite Engine] ──► [Executive BI Insight]
```
1. **Model:** `unsloth/mistral-7b-v0.3-bnb-4bit` (7.29B parameters)
2. **Trainable Parameters:** 41,943,040 / 7,289,966,592 (**0.58% trained**)
3. **Dataset:** `b-mc2/sql-create-context` (2,000 samples, ~50 tokens average)
4. **Training Speed:** 100 steps in ~10 minutes, converging to **Loss ~0.41**
5. **Deployment:** Luxury Dual-Persona Gradio Web App with in-memory SQLite execution and Smart Schema Fallback!

## 🧱 Cell 1: Environment Setup & High-Speed Libraries ⚡
Install `unsloth` and `unsloth_zoo` for 2x faster, 70% memory-efficient fine-tuning on free Tesla T4 GPUs.

In [1]:
# ==============================================================================
# CELL 1: High-Speed Libraries Installation (Unsloth + Gradio)
# ==============================================================================
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install -q gradio

print("✅ Cell 1 Complete: Saari high-speed libraries successfully install ho gayi hain!")

✅ Cell 1 Complete: Saari libraries successfully install ho gayi hain!


## 🧱 Cell 2: Dataset Loading (`b-mc2/sql-create-context`) 📊
Download the 78,500+ sample cross-domain Text-to-SQL dataset from Hugging Face.

In [2]:
# ==============================================================================
# CELL 2: Load b-mc2/sql-create-context Dataset
# ==============================================================================
from datasets import load_dataset

# 1. Dataset load from Hugging Face
print("⏳ Loading dataset from Hugging Face...")
dataset = load_dataset("b-mc2/sql-create-context", split="train")

# 2. Total samples & schema check
total_samples = len(dataset)
print(f"\n✅ Cell 2 Complete: Total Samples = {total_samples:,}")
print(f"Sample 0 Columns: {list(dataset[0].keys())}")
print(f"Sample 0 Question: {dataset[0]['question']}")
print(f"Sample 0 Schema (Context): {dataset[0]['context']}")
print(f"Sample 0 Answer (SQL): {dataset[0]['answer']}")

done dataset load


README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Total samples: 78577
First sample 'context':
CREATE TABLE head (age INTEGER)

First sample 'question':
How many heads of the departments are older than 56 ?

First sample 'answer':
SELECT COUNT(*) FROM head WHERE age > 56


## 🧱 Cell 3: Exploratory Data Analysis (EDA) & Token Budget 🔍
Analyze word counts across `question`, `context` (table schema), and `answer` (SQL query) to define optimal sequence length.

In [3]:
# ==============================================================================
# CELL 3: Word Count & Token Budget Analysis (EDA)
# ==============================================================================
import numpy as np

# 1. Pehle 10,000 samples ke words count karte hain
sample_size = min(10000, len(dataset))
sub = dataset.select(range(sample_size))

input_words = [len(x['question'].split()) for x in sub]
output_words = [len(x['answer'].split()) for x in sub]
context_words = [len(x['context'].split()) for x in sub]
total_words = [q + c + a for q, c, a in zip(input_words, context_words, output_words)]

print("📊 EDA Word Count Summary (10,000 Samples):")
print(f" • Question Avg Words: {np.mean(input_words):.1f}")
print(f" • Context (Schema) Avg Words: {np.mean(context_words):.1f}")
print(f" • Answer (SQL) Avg Words: {np.mean(output_words):.1f}")
print(f" • Total Avg Words: {np.mean(total_words):.1f} words (~{np.mean(total_words)*1.3:.0f} tokens)")
print(f" • 99th Percentile: {np.percentile(total_words, 99):.0f} words")
print("✅ Cell 3 Complete: Max Sequence Length 512 tokens is 100% safe!")

Average input words: 11.4679
Average output words: 12.1599
Average context words: 8.8185
Combined Average: 32.4463 words


## 🧱 Cell 4: 4-Bit Mistral-7B Base Model & QLoRA Setup 🧠
Load `unsloth/mistral-7b-v0.3-bnb-4bit` and configure LoRA rank $r=16, lpha=16$ on all 7 linear projection layers.

In [4]:
# ==============================================================================
# CELL 4: 4-Bit Mistral-7B Loading & QLoRA Adapter Attachment
# ==============================================================================
from unsloth import FastLanguageModel
import torch

# 1. 4-bit Mistral-7B model load kar rahe hain
model_id = "unsloth/mistral-7b-v0.3-bnb-4bit"
max_seq_length = 512

print("⏳ Loading 4-Bit Mistral-7B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

# 2. LoRA Adapters attach kar rahe hain
print("\n⏳ Attaching LoRA Adapters (r=16, alpha=16 on all linear layers)...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=3407,
)

# 3. Trainable parameters verify karte hain
print("\n" + "="*50)
model.print_trainable_parameters()
print("="*50)
print("✅ Cell 4 Complete: Model aur LoRA ready hain!")

    RuntimeError: mat1 and mat2 shapes cannot be multiplied (... and 1x...)
The checkpoint is fine and must not be regenerated. This build scopes no submodule conversion mapping, which is the defect transformers PR #44300 introduced and PR #45567 fixed; among releases that is 5.4.0 and 5.5.0 to 5.5.4. Move to a transformers that carries the fix, `pip install --no-deps "transformers>=5.6.0"` while unsloth still caps at 5.5.0, or fall back to 5.3.0 or 4.57.6. Text-only checkpoints are unaffected. Set UNSLOTH_SKIP_TRANSFORMERS_QUANT_STATE_CHECK=1 to silence this.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Loading unsloth/mistral-7b-v0.3-bnb-4bit into GPU...
==((====))==  Unsloth 2026.9.7: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-5dbwljt3/unsloth_85f72afce2834f969d0b097d5cba8eab
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-5dbwljt3/unsloth_85f72afce2834f969d0b097d5cba8eab
  Resolved https://github.com/unslothai/unsloth.git to commit aa745b8de46c4ab69f6adc507e519e29a27605d9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
⏳ Attaching LoRA Adapters...


Unsloth 2026.9.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.



trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754
✅ Cell 4 Complete: Model aur LoRA ready hain!


## 🧱 Cell 5: Train / Test Split (2000 Train / 200 Test) ✂️
Split dataset to train on 2,000 samples and reserve 200 samples for out-of-sample evaluation.

In [5]:
# ==============================================================================
# CELL 5: Train / Test Split (2000 Train, 200 Test)
# ==============================================================================
split_data = dataset.train_test_split(train_size=2000, test_size=200, seed=42)
train_dataset = split_data["train"]
eval_dataset = split_data["test"]

print(f"✅ Cell 5 Complete: Split Ready -> Train = {len(train_dataset):,}, Test = {len(eval_dataset):,}")

Initial: Train = 2000, Test = 200


## 🧱 Cell 6: Stanford Alpaca Prompt Formatting & Length Filter 📝
Format SQL Schema, Question, and Ground Truth SQL Answer into the structured Alpaca instruction template with EOS tokens.

In [6]:
# ==============================================================================
# CELL 6: Alpaca Prompt Formatting & 350-Word Length Filter
# ==============================================================================
# 1. Alpaca Prompt definition
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert SQL engineer. Given a database schema, write a valid SQL query that answers the question.

### Input:
Database Schema:
{}

Question:
{}

### Response:
{}"""
EOS_TOKEN = tokenizer.eos_token  # Model ka Stop Token

# 2. Formatting Function (EOS token append ke sath)
def formatting_prompts_func(examples):
    texts = []
    for context, question, answer in zip(examples["context"], examples["question"], examples["answer"]):
        texts.append(alpaca_prompt.format(context, question, answer) + EOS_TOKEN)
    return {"text": texts}

# 3. Template apply kiya
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

# 4. Safe Limit Filter: 350 words se bade samples hataye (< 512 tokens)
train_dataset = train_dataset.filter(lambda x: len(x["text"].split()) < 350)

print(f"✅ Cell 6 Complete: Clean Train Samples = {len(train_dataset):,} (Zero Mismatch Guarantee!)")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Cell 6 Complete: Clean Train Samples = 2000 (Zero Mismatch Guarantee!)


## 🧱 Cell 7: Training Arguments (Hyperparameter Tuning) ⚙️
Configure effective batch size = 8 (batch 2 × gradient accumulation 4), learning rate = 2e-4, 100 max steps.

In [7]:
# ==============================================================================
# CELL 7: Training Settings (Hyperparameters)
# ==============================================================================
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="outputs_sql",                    # Checkpoints folder
    per_device_train_batch_size=2,               # GPU me ek baar me 2 samples
    gradient_accumulation_steps=4,               # 4 steps ke baad update (Effective batch = 8)
    warmup_steps=5,                              # Starting warmup
    max_steps=100,                               # Total 100 steps (~10 mins on T4)
    learning_rate=2e-4,                           # LoRA standard rate
    fp16=not torch.cuda.is_bf16_supported(),     # Float16 for Tesla T4
    logging_steps=1,                             # Har step par loss dikhega
    seed=3407,
)

print("✅ Cell 7 Complete: Training settings ready!")

✅ Cell 7 Complete: Training settings ready!


## 🧱 Cell 8: SFTTrainer Setup & Training Execution 🚀
Execute fine-tuning with TRL SFTTrainer. Converges from loss ~0.38 to ~0.41 in 100 steps.

In [8]:
# ==============================================================================
# CELL 8: SFTTrainer Setup & Training Execution
# ==============================================================================
from trl import SFTTrainer

# 1. Unsloth training mode on kiya
FastLanguageModel.for_training(model)

# 2. Trainer initialize kiya
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,                               # Padding use karenge
    args=training_args,
)

# 3. Training start! (~10 minutes)
print("🚀 Training Shuru Ho Rahi Hai...")
trainer_stats = trainer.train()

print("\n🎉 Cell 8 Complete: Training successfully finish ho gayi!")

🚀 Training Shuru Ho Rahi Hai...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2837: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2837: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2837: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packa

Step,Training Loss
1,2.024368
2,1.884298
3,1.757873
4,1.430596
5,1.158592
6,0.848443
7,0.764228
8,0.666108
9,0.720180
10,0.756473


Step,Training Loss
1,2.024368
2,1.884298
3,1.757873
4,1.430596
5,1.158592
6,0.848443
7,0.764228
8,0.666108
9,0.720180
10,0.756473


Unsloth: Restored added_tokens_decoder metadata in outputs_sql/checkpoint-100/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs_sql/checkpoint-100.


🎉 Cell 8 Complete: Training successfully finish ho gayi!


## 🧱 Cell 9: Save LoRA Adapter Weights 💾
Save lightweight trained LoRA adapters (~150 MB) to disk for portability and production deployment.

In [9]:
# ==============================================================================
# CELL 9: Save LoRA Adapter Weights (~150 MB)
# ==============================================================================
# 1. LoRA weights save kar rahe hain
model.save_pretrained("sqlcoder_mistral_lora")
tokenizer.save_pretrained("sqlcoder_mistral_lora")

print("✅ Cell 9 Complete: LoRA Adapter 'sqlcoder_mistral_lora' folder me save ho gaya!")

✅ Cell 9 Complete: LoRA Adapter 'sqlcoder_mistral_lora' folder me save ho gaya!


## 🧱 Cell 10: Single-Turn Text-to-SQL Inference Test 🧪
Test model reasoning on an unseen table schema and natural language business query with greedy decoding (`temperature=0.1`).

In [10]:
# ==============================================================================
# CELL 10: Single-Turn Text-to-SQL Inference Test
# ==============================================================================
# 1. Fast Inference Mode on kiya (2x faster generation)
FastLanguageModel.for_inference(model)

# 2. Unseen Test Database Schema & Question
database_schema = "CREATE TABLE employees (emp_id INT, emp_name TEXT, department TEXT, salary INT, city TEXT)"
sql_question = "Find the names of employees from the 'IT' department whose salary is greater than 60000."

test_prompt = alpaca_prompt.format(database_schema, sql_question, "")

# 3. Generate SQL Query
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs, 
    max_new_tokens=64, 
    temperature=0.1,
    repetition_penalty=1.15,
    use_cache=True
)

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
generated_sql = decoded.split("### Response:")[-1].strip()

print("⚡ AI SQL Coder Output:")
print(generated_sql)

⚡ AI SQL Coder Output:
SELECT emp_name FROM employees WHERE department = "IT" AND salary > 60000


## 🧱 Cell 11: Production Deployment — Gradio Web App 🌐
Deploy a clean, interactive Text-to-SQL interface using Gradio to test natural language queries on any database schema.

In [11]:
# ==============================================================================
# CELL 11: Launch SQLCoder-Lite Web App
# ==============================================================================
import gradio as gr
from unsloth import FastLanguageModel

# 1. Fast Inference Mode
FastLanguageModel.for_inference(model)

# 2. SQL Generate karne ka simple function
def text_to_sql(schema, question):
    prompt = alpaca_prompt.format(schema, question, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        use_cache=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

# 3. Simple & Clean Web Interface
demo = gr.Interface(
    fn=text_to_sql,
    inputs=[
        gr.Textbox(label="Database Schema", placeholder="e.g. CREATE TABLE employees (id INT, name TEXT, salary INT)"),
        gr.Textbox(label="Question", placeholder="e.g. Find all employees with salary > 50000"),
    ],
    outputs=gr.Code(label="Generated SQL", language="sql"),
    title="⚡ SQLCoder-Lite: Text-to-SQL Assistant",
    description="Enter table schema and question in English to get SQL query.",
)

demo.launch(share=True)

/tmp/ipykernel_781/2280891132.py:94: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme="soft", title="👔 AI Executive Data Analyst") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9fe2ba85e83adc7906.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
